# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhinav77040/Flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Rule

I will prioritize webpages that have not been updated for a long time but still receive meaningful traffic. These pages have a higher chance of improving SEO if refreshed.

### Reason codes

- STALE_HIGH_TRAFFIC – Old page with high clicks.
- STALE_MEDIUM_TRAFFIC – Old page with moderate clicks.
- LOW_PRIORITY – Recent page or very low traffic.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
import os
import pandas as pd
import numpy as np

# Staleness buckets
bins = [0,30,90,180,365,10000]

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=bins
)

table1 = df.groupby("staleness_bucket").agg(
    n=("content_id","count"),
    avg_clicks=("clicks_90d","mean")
)

print(table1)

# Second signal
table2 = df.groupby("position_tier").agg(
    n=("content_id","count"),
    avg_ctr=("ctr","mean")
)

print(table2)

# Baseline score
df["baseline_score"] = (
    df["days_since_last_update"].fillna(0) * 0.6 +
    df["clicks_90d"].fillna(0) * 0.4
)

# Reason codes
conditions = [
    (df["days_since_last_update"] > 180) & (df["clicks_90d"] > 100),
    (df["days_since_last_update"] > 90)
]

choices = [
    "STALE_HIGH_TRAFFIC",
    "STALE_MEDIUM_TRAFFIC"
]

df["reason_code"] = np.select(
    conditions,
    choices,
    default="LOW_PRIORITY"
)

# Action label
df["action"] = np.where(
    df["baseline_score"] >= df["baseline_score"].median(),
    "REFRESH",
    "MONITOR"
)

# Rank queue
ranked = df.sort_values(
    "baseline_score",
    ascending=False
)

os.makedirs("work/outputs", exist_ok=True)

ranked.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV written successfully.")

ranked.head(20)

/tmp/ipykernel_1059/2702589134.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  table1 = df.groupby("staleness_bucket").agg(


                      n  avg_clicks
staleness_bucket                   
(0, 30]           20480   13.727393
(30, 90]            175    9.685714
(90, 180]          9171   21.766765
(180, 365]          169    2.745562
(365, 10000]          5    0.200000
                   n   avg_ctr
position_tier                 
deep            1319  0.150212
page_1         11814  0.652467
page_3_5        7242  0.222484
striking        7304  0.323239
top_3           2321  1.483611
CSV written successfully.


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,staleness_bucket,baseline_score,reason_code,action
10741,content_07e0b9af8b1a,client_b4944c6ff0,20.0,0.71,HIGH,0.13,keyword article,transactional,2898.0,20836.0,...,8.03,0.03,excellent,page_1,down,-39.3,"(0, 30]",1676.0,LOW_PRIORITY,REFRESH
21565,content_9532f197bbc8,client_4e07408562,10.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,28.75,0.00,excellent,top_3,down,-37.3,"(90, 180]",1138.0,STALE_MEDIUM_TRAFFIC,REFRESH
16811,content_8e7ba84a972b,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3731.0,23829.0,...,37.88,0.00,excellent,page_1,stable,-18.1,"(0, 30]",1073.2,LOW_PRIORITY,REFRESH
14234,content_89e84d699e9e,client_349c41201b,50.0,0.05,LOW,0.08,keyword article,informational,3480.0,22147.0,...,1.08,0.00,excellent,page_1,stable,-14.8,"(0, 30]",996.0,LOW_PRIORITY,REFRESH
2797,content_c8ad1f4d0e56,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,transactional,3418.0,23976.0,...,42.79,0.10,excellent,page_1,down,-35.2,"(0, 30]",867.2,LOW_PRIORITY,REFRESH
14090,content_44e481c8f55b,client_19581e27de,20.0,0.26,LOW,0.43,keyword article,transactional,3040.0,21293.0,...,2.78,0.00,excellent,top_3,stable,-11.6,"(0, 30]",824.8,LOW_PRIORITY,REFRESH
13537,content_2c2606c5d176,client_19581e27de,590.0,0.18,LOW,0.31,keyword article,commercial,NaN,NaN,...,3.28,0.00,excellent,page_1,down,-36.5,"(90, 180]",804.0,STALE_MEDIUM_TRAFFIC,REFRESH
21819,content_4c36c775b818,client_4e07408562,40.0,0.00,LOW,0.00,keyword article,informational,3097.0,20514.0,...,18.07,0.00,excellent,top_3,down,-33.2,"(0, 30]",767.6,LOW_PRIORITY,REFRESH
19645,content_3ad3a781fa91,client_4e07408562,0.0,0.00,LOW,0.00,keyword article,informational,1725.0,11151.0,...,9.59,0.26,excellent,page_1,stable,-15.7,"(90, 180]",688.0,STALE_MEDIUM_TRAFFIC,REFRESH
14178,content_f3e5505473ad,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,6611.0,43764.0,...,3.43,0.23,excellent,page_1,stable,-7.5,"(90, 180]",625.6,STALE_MEDIUM_TRAFFIC,REFRESH


The highest ranked pages are recommended for REFRESH because they combine high staleness with strong historical traffic.

Reason code:
- Mostly STALE_HIGH_TRAFFIC
- Some STALE_MEDIUM_TRAFFIC

Confidence:
Medium. The ranking is based on historical engagement only.

What would make these recommendations wrong?

- Seasonal traffic
- Recent unpublished updates
- Missing or inaccurate analytics
- Pages intentionally left unchanged

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some lower-ranked pages may still deserve a refresh if they have business importance that is not captured by traffic.

Leakage check:

- No future clicks or impressions were used.
- Only historical signals were used.
- No product labels or future outcomes were included.
- The rule is decision-support only and does not predict future Google rankings.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.